# Alpaca Golden Cross Stock Screener

Run the full screener from a notebook: fetch Alpaca daily bars, find recent golden crosses with volume confirmation, calculate support distance, sort closest-to-support first, and save results.

## 1. Setup

Install the project first from the repository root:

```powershell
pip install -e ".[notebook]"
```

Copy `.env.example` to `.env` and add your Alpaca credentials before running the data cells.

In [ ]:
from pathlib import Path
import inspect
import sys

try:
    import pandas as pd
    from dotenv import load_dotenv
    from alpaca.data.enums import DataFeed
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing notebook dependencies. From the repository root, run: "
        'pip install -e ".[notebook]"'
    ) from exc

repo_root = Path.cwd()
if not (repo_root / "alpaca_screener").exists():
    repo_root = repo_root.parent
repo_root = repo_root.resolve()

# Prefer this checkout over any installed or previously imported copy.
repo_root_text = str(repo_root)
if repo_root_text in sys.path:
    sys.path.remove(repo_root_text)
sys.path.insert(0, repo_root_text)
for module_name in list(sys.modules):
    if module_name == "alpaca_screener" or module_name.startswith("alpaca_screener."):
        del sys.modules[module_name]

from alpaca_screener.alpaca_data import (
    clients_from_environment,
    fetch_daily_bars,
    get_tradable_symbols,
)
from alpaca_screener.strategy import ScreenConfig, screen_bars
from alpaca_screener.universe import (
    UniverseFilterConfig,
    load_fundamentals_file,
    filter_universe,
    volume_stats_from_bars,
)

if "ma_type" not in inspect.signature(ScreenConfig).parameters:
    raise RuntimeError(
        "Loaded an old ScreenConfig. Restart the kernel, then rerun this setup cell."
    )

load_dotenv(repo_root / ".env")
repo_root

## 2. Configure the screen

Use `SYMBOLS` for a fast watchlist run, or set it to an empty list to screen active tradable US equities. While experimenting, keep `LIMIT_UNIVERSE` small.

In [ ]:
if "ma_type" not in inspect.signature(ScreenConfig).parameters:
    raise RuntimeError(
        "The notebook kernel has an old alpaca_screener import loaded. "
        "Rerun the Setup cell above, or restart the kernel and run all cells."
    )

SYMBOLS = []
LIMIT_UNIVERSE = 500  # Example: 500 when SYMBOLS = []
TOP = 25
FEED = DataFeed.IEX  # Use DataFeed.SIP only when your Alpaca account has SIP access.
FUNDAMENTALS_FILE = None  # Example: repo_root / "data" / "fundamentals.csv"

universe_config = UniverseFilterConfig(
    fundamentals_file=str(FUNDAMENTALS_FILE) if FUNDAMENTALS_FILE else None,
    pe_min=None,
    pe_max=None,
    industries=(),  # Example: ("Semiconductors", "Technology")
    market_caps=(),  # Any of: "small", "mid"/"medium", "large"
    cap_mix={"large": 50},  # Example: {"small": 20, "mid": 30, "large": 50}
    min_30d_share_volume=None,
    min_30d_dollar_volume=None,
    performance_windows=("3m",),  # Any of: "1m", "3m", "6m"
    top_performance_pct=2.0,  # Example: top 2% of performers
    min_performance={},  # Example: {"3m": 25.0}
    max_symbols=None,
)

config = ScreenConfig(
    fast_window=50,
    slow_window=200,
    ma_type="sma",  # "sma" or "ema"
    crossover_lookback=5,
    volume_window=20,
    volume_spike_lookback=3,
    volume_spike_pct=50.0,
    support_lookback=90,
    support_pivot_span=3,
    support_modes=("pivot", "recent_low", "1_month_low"),
    support_lookbacks=(),  # Example: (21, 63, 126)
    min_price=5.0,
    min_average_volume=100_000.0,
)
config, universe_config

## 3. Fetch Alpaca data

In [ ]:
data_client, trading_client = clients_from_environment()

if SYMBOLS:
    base_symbols = [symbol.upper() for symbol in SYMBOLS]
else:
    base_symbols = get_tradable_symbols(trading_client, LIMIT_UNIVERSE)

clock = trading_client.get_clock()
incomplete_session_date = clock.timestamp.date() if clock.is_open else None

if universe_config.needs_fundamentals and not universe_config.fundamentals_file:
    raise RuntimeError(
        "Set FUNDAMENTALS_FILE to a CSV before using P/E, market-cap, cap-mix, "
        "or industry filters. This notebook no longer calls yfinance per symbol."
    )
fundamentals = (
    load_fundamentals_file(universe_config.fundamentals_file)
    if universe_config.needs_fundamentals
    else None
)
volume_stats = None
if universe_config.needs_volume:
    volume_bars = fetch_daily_bars(
        data_client,
        base_symbols,
        calendar_days=50,
        feed=FEED,
        incomplete_session_date=incomplete_session_date,
    )
    volume_stats = volume_stats_from_bars(volume_bars)

symbols = filter_universe(
    base_symbols,
    universe_config,
    fundamentals=fundamentals,
    volume_stats=volume_stats,
)
bars_by_symbol = fetch_daily_bars(
    data_client,
    symbols,
    feed=FEED,
    incomplete_session_date=incomplete_session_date,
)

len(base_symbols), len(symbols), len(bars_by_symbol)

## 4. Run the screener

Results are sorted by `distance_to_support_pct` from closest to furthest.

In [ ]:
results = screen_bars(bars_by_symbol, config).head(TOP)
results

## 5. Save results

In [ ]:
output_dir = repo_root / "outputs"
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "screen-results.csv"
json_path = output_dir / "screen-results.json"

results.to_csv(csv_path, index=False)
results.to_json(json_path, orient="records", indent=2)

csv_path, json_path

## 6. Optional: chart the top match

This chart shows close, the configured fast/slow moving averages, and calculated support for the closest-to-support match.

In [ ]:
import matplotlib.pyplot as plt

if results.empty:
    print("No matches to chart.")
else:
    symbol = results.iloc[0]["symbol"]
    support = results.iloc[0]["support"]
    chart = bars_by_symbol[symbol].sort_index().tail(260).copy()

    if config.ma_type.lower() == "ema":
        chart["ma_fast"] = chart["close"].ewm(
            span=config.fast_window, adjust=False, min_periods=config.fast_window
        ).mean()
        chart["ma_slow"] = chart["close"].ewm(
            span=config.slow_window, adjust=False, min_periods=config.slow_window
        ).mean()
    else:
        chart["ma_fast"] = chart["close"].rolling(config.fast_window).mean()
        chart["ma_slow"] = chart["close"].rolling(config.slow_window).mean()

    ax = chart[["close", "ma_fast", "ma_slow"]].plot(
        figsize=(12, 6), title=f"{symbol} ({config.ma_type.upper()})"
    )
    ax.axhline(support, color="tab:green", linestyle="--", label="support")
    ax.set_ylabel("Price")
    ax.legend()
    plt.show()